In [ ]:
import numpy as np
import pandas as pd

import optuna

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(
    "../data/processed/final_features_v1.csv"
)

TARGET = "SeriousDlqin2yrs"


X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


print(X.shape)
print(y.value_counts())

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
models = {

    "CatBoost": CatBoostClassifier(
        verbose=0,
        random_state=42
    ),


    "LightGBM": LGBMClassifier(
        random_state=42
    ),


    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    )

}

In [ ]:
param_spaces = {


"CatBoost": {

    "iterations": ("int",300,1000),

    "learning_rate": ("logfloat",0.01,0.2),

    "depth": ("int",4,10),

    "l2_leaf_reg": ("float",1,10),

    "bagging_temperature": ("float",0,5)

},



"LightGBM": {

    "n_estimators": ("int",300,1000),

    "learning_rate": ("logfloat",0.01,0.2),

    "num_leaves": ("int",16,256),

    "max_depth": ("int",3,12),

    "min_child_samples": ("int",10,100),

    "reg_alpha": ("float",0,10),

    "reg_lambda": ("float",0,10)

},



"XGBoost": {

    "n_estimators": ("int",300,1000),

    "learning_rate": ("logfloat",0.01,0.2),

    "max_depth": ("int",3,10),

    "min_child_weight": ("int",1,20),

    "subsample": ("float",0.6,1),

    "colsample_bytree": ("float",0.6,1),

    "gamma": ("float",0,5),

    "reg_alpha": ("float",0,10),

    "reg_lambda": ("float",0,10)

}

}

In [ ]:
def tune_model(
    model_name,
    model,
    param_space,
    trials=20
):


    def objective(trial):


        params = {}


        for key, value in param_space.items():

            p_type = value[0]


            if p_type == "int":

                params[key] = trial.suggest_int(
                    key,
                    value[1],
                    value[2]
                )


            elif p_type == "logfloat":

                params[key] = trial.suggest_float(
                    key,
                    value[1],
                    value[2],
                    log=True
                )


            else:

                params[key] = trial.suggest_float(
                    key,
                    value[1],
                    value[2]
                )



        fold_scores = []



        for train_idx, valid_idx in skf.split(X,y):


            X_train = X.iloc[train_idx]
            X_valid = X.iloc[valid_idx]


            y_train = y.iloc[train_idx]
            y_valid = y.iloc[valid_idx]



            # create fresh model
            current_model = clone(model)


            # apply trial parameters
            current_model.set_params(
                **params
            )


            current_model.fit(
                X_train,
                y_train
            )



            probs = current_model.predict_proba(
                X_valid
            )[:,1]



            score = average_precision_score(
                y_valid,
                probs
            )


            fold_scores.append(score)



        return np.mean(fold_scores)



    study = optuna.create_study(
        direction="maximize"
    )


    study.optimize(
        objective,
        n_trials=trials
    )


    print("\n==============================")
    print(model_name)
    print("Best PR-AUC:", study.best_value)
    print("Best Parameters:")
    print(study.best_params)


    return study

In [ ]:
studies = {}


for name in models:

    studies[name] = tune_model(
        name,
        models[name],
        param_spaces[name],
        trials=20
    )

In [ ]:
results = []


for name, study in studies.items():

    results.append({

        "Model": name,

        "Best_PR_AUC": study.best_value,

        "Best_Parameters": study.best_params

    })


tuning_results = pd.DataFrame(results)


tuning_results = tuning_results.sort_values(
    by="Best_PR_AUC",
    ascending=False
).reset_index(drop=True)


tuning_results.insert(
    0,
    "Rank",
    range(1, len(tuning_results)+1)
)


tuning_results

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


def evaluate_tuned_model(model_name, model):

    precision_scores = []
    recall_scores = []
    f1_scores = []
    roc_scores = []
    pr_scores = []


    for train_idx, valid_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]


        model.fit(
            X_train,
            y_train
        )


        probs = model.predict_proba(
            X_valid
        )[:,1]


        preds = (
            probs >= 0.5
        ).astype(int)


        precision_scores.append(
            precision_score(
                y_valid,
                preds
            )
        )


        recall_scores.append(
            recall_score(
                y_valid,
                preds
            )
        )


        f1_scores.append(
            f1_score(
                y_valid,
                preds
            )
        )


        roc_scores.append(
            roc_auc_score(
                y_valid,
                probs
            )
        )


        pr_scores.append(
            average_precision_score(
                y_valid,
                probs
            )
        )


    return {

        "Model": model_name,

        "ROC-AUC": np.mean(roc_scores),

        "PR-AUC": np.mean(pr_scores),

        "Precision": np.mean(precision_scores),

        "Recall": np.mean(recall_scores),

        "F1": np.mean(f1_scores)

    }

In [ ]:
tuned_models = {

    "CatBoost Tuned": CatBoostClassifier(
        **studies["CatBoost"].best_params,
        verbose=0,
        random_state=42
    ),


    "LightGBM Tuned": LGBMClassifier(
        **studies["LightGBM"].best_params,
        random_state=42
    ),


    "XGBoost Tuned": XGBClassifier(
        **studies["XGBoost"].best_params,
        random_state=42,
        eval_metric="logloss"
    )

}

In [ ]:
evaluation_results = []


for name, model in tuned_models.items():

    evaluation_results.append(
        evaluate_tuned_model(
            name,
            model
        )
    )


evaluation_df = pd.DataFrame(
    evaluation_results
)


evaluation_df.sort_values(
    by="PR-AUC",
    ascending=False
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

from catboost import CatBoostClassifier
import numpy as np
import pandas as pd


best_catboost = CatBoostClassifier(
    **studies["CatBoost"].best_params,
    verbose=0,
    random_state=42
)


oof_probs = np.zeros(len(X))


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]


    model = CatBoostClassifier(
        **studies["CatBoost"].best_params,
        verbose=0,
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    oof_probs[valid_idx] = model.predict_proba(
        X_valid
    )[:,1]

In [ ]:
threshold_results = []


thresholds = np.arange(
    0.05,
    0.95,
    0.01
)


for threshold in thresholds:

    preds = (
        oof_probs >= threshold
    ).astype(int)


    precision = precision_score(
        y,
        preds
    )


    recall = recall_score(
        y,
        preds
    )


    f1 = f1_score(
        y,
        preds
    )


    threshold_results.append({

        "Threshold": round(threshold,2),

        "Precision": precision,

        "Recall": recall,

        "F1": f1

    })


threshold_df = pd.DataFrame(
    threshold_results
)


threshold_df

In [ ]:
acceptable_thresholds = threshold_df[
    (threshold_df["Precision"] >= 0.70) &
    (threshold_df["Recall"] >= 0.85)
]


acceptable_thresholds

In [ ]:
if acceptable_thresholds.empty:

    print(
        "Threshold optimization failed."
    )

    print(
        "Move to WOE + Logistic Regression."
    )

else:

    best_threshold = (
        acceptable_thresholds
        .sort_values(
            "F1",
            ascending=False
        )
        .iloc[0]
    )

    print(best_threshold)